# Tokenizador fonémico del maya yucateco

`mayanlab.tokenizer.MayaPhonemeTokenizer` convierte ortografía maya en fonemas del
inventario Kaldi. Este notebook comprueba su inventario, su cobertura sobre el corpus
y qué hace con los préstamos del español.

**Dependencia del sistema:** las palabras que el tokenizador maya no puede fonemizar caen
a `phonemizer` con backend **espeak**, que necesita `espeak-ng` instalado
(`sudo apt install espeak-ng`). Ver `tools/README.md`.

In [1]:
from mayanlab.tokenizer import IPA_TO_KALDI, MayaPhonemeTokenizer

tokenizer = MayaPhonemeTokenizer()

## Inventario de fonemas

In [2]:
# `phones` une los fonemas de la tabla g2p maya con los que salen de remapear IPA
# (los que aparecen al fonemizar préstamos del español).
print(f"grafemas en la tabla g2p: {len(tokenizer.g2p_dict)}")
print(f"fonemas del inventario:   {len(tokenizer.phones)}\n")
print(" ".join(tokenizer.phones))

grafemas en la tabla g2p: 47
fonemas del inventario:   53

A AA_H AA_L A_G A_RG B CH CH_G D E EE_H EE_L E_G E_RG F G GLOT I II_H II_L I_G I_RG J K K_G L LY M N NG NY O OO_H OO_L O_G O_RG P P_G R RR S T TS TS_G T_G U UU_H UU_L U_G U_RG W X Y


In [3]:
# De dónde sale cada fonema del inventario
solo_maya = sorted(set(tokenizer.g2p_dict.values()) - set(IPA_TO_KALDI.values()))
solo_es   = sorted(set(IPA_TO_KALDI.values()) - set(tokenizer.g2p_dict.values()))
comunes   = sorted(set(tokenizer.g2p_dict.values()) & set(IPA_TO_KALDI.values()))

print(f"solo del maya ({len(solo_maya)}):", " ".join(solo_maya))
print(f"solo del español vía IPA ({len(solo_es)}):", " ".join(solo_es))
print(f"comunes ({len(comunes)}):", " ".join(comunes))

solo del maya (25): AA_H AA_L A_G A_RG CH_G EE_H EE_L E_G E_RG II_H II_L I_G I_RG K_G OO_H OO_L O_G O_RG P_G TS_G T_G UU_H UU_L U_G U_RG
solo del español vía IPA (11): A I A U E I E U F G LY NG NY O I RR
comunes (22): A B CH D E GLOT I J K L M N O P R S T TS U W X Y


## Tokenización palabra a palabra

In [4]:
for palabra in ["ko'olel", "xta'akumbil", "ch'íich'", "k'áat", "jump'éel", "ya'axche"]:
    print(f"{palabra:<14} -> {tokenizer.tokenize(palabra)}")

ko'olel        -> K O_RG L E L
xta'akumbil    -> X T A_RG K U M B I L
ch'íich'       -> CH_G II_H CH_G
k'áat          -> K_G AA_H T
jump'éel       -> J U M P_G EE_H L
ya'axche       -> Y A_RG X CH E


## Préstamos del español

`tokenize` intenta primero la tabla g2p maya. Si la palabra tiene símbolos fuera del
inventario, cae a `phonemizer`/espeak en español y remapea el IPA resultante con
`IPA_TO_KALDI`. `SIL` separa palabras.

In [5]:
frase = "jach máan dyos bo'otik don alfonso"
print(frase)
print(tokenizer.tokenize(frase))

jach máan dyos bo'otik don alfonso
J A CH SIL M AA_H N SIL D Y O S SIL B O_RG T I K SIL D O N SIL A L F O N S O


In [6]:
# Qué palabras de la frase NO son fonemizables por la tabla maya (y caen a espeak)
for w in frase.split():
    directa = tokenizer._try_tokenize_word(w)
    origen = "g2p maya" if directa is not None else "espeak (español)"
    print(f"{w:<10} {origen:<18} {tokenizer.tokenize(w)}")

jach       g2p maya           J A CH
máan       g2p maya           M AA_H N
dyos       g2p maya           D Y O S
bo'otik    g2p maya           B O_RG T I K
don        g2p maya           D O N


alfonso    espeak (español)   A L F O N S O


In [7]:
# Backends de phonemizer.
#
# El tokenizador usa **espeak**, que es el que soporta español. `festival` solo
# fonemiza inglés: la celda original de este notebook pedía `language="es"` con
# backend festival y por eso nunca pudo funcionar.
from phonemizer import phonemize
from phonemizer.backend import BACKENDS

for nombre in ("espeak", "festival"):
    backend = BACKENDS[nombre]
    disponible = backend.is_available()
    idiomas = sorted(backend.supported_languages())
    print(f"{nombre:<10} disponible={disponible}  idiomas={len(idiomas)}  "
          f"¿español? {'es' in idiomas}")

print()
print("espeak   es  :", phonemize("gato", language="es", backend="espeak").strip())
print("festival en-us:", phonemize("cat", language="en-us", backend="festival").strip())

espeak     disponible=True  idiomas=130  ¿español? True
festival   disponible=True  idiomas=1  ¿español? False



espeak   es  : ɡato


festival en-us: kaet


## Cobertura sobre el vocabulario del corpus

In [8]:
import pandas as pd

from mayanlab.paths import MANIFESTS

df = pd.read_csv(MANIFESTS / "final_dataset.csv")
vocabulario = sorted({w for frase in df["maya"].astype(str) for w in frase.lower().split()})

directas = [w for w in vocabulario if tokenizer._try_tokenize_word(w) is not None]
fallback = [w for w in vocabulario if tokenizer._try_tokenize_word(w) is None]

print(f"palabras únicas en el corpus: {len(vocabulario)}")
print(f"  fonemizables por la tabla maya: {len(directas)} ({100*len(directas)/len(vocabulario):.1f} %)")
print(f"  caen a espeak:                  {len(fallback)} ({100*len(fallback)/len(vocabulario):.1f} %)")
if fallback:
    print("\nejemplos que caen a espeak:", ", ".join(fallback[:25]))

palabras únicas en el corpus: 5802
  fonemizables por la tabla maya: 4993 (86.1 %)
  caen a espeak:                  809 (13.9 %)

ejemplos que caen a espeak: $25, $50, 134, aah, acabó, aceptartik, aceptartiko'obe, administración, adolfo, advertenkia, afuera, agosto, agua, ahora, akompañartiken, albañil, alcanzartaj, alcanzarte, algo, aliviar, allá, amigo, angelitoe, animás, antiguos


## Sobre un fragmento real de narración

In [9]:
fragmento = """xtakumbil xunáan le tsikbalo u k'aaba'e xta'akumbil xunáan xta'akumbil xunáane u k'áat u ya'ale tu'ux ta'aka'an jump'éel ko'olel jaaj ta'akumbil xunáano le xunáano jump'éel ko'olel le u k'áat u ya'al beyo lelo yaan jump'éel le chan kaajo u k'aaba'e ya'axche palomeke u ya'ala'al ti u k'aaba le chan kaajo entonkes beyo lelo le ko'olelo sáansamale yaan u bin xíinxíimbal ti le chan kaajo káan u yila tu bin tu taal u yáa'biltale ku yokol ti le chan kaajo ku máan u k'áat tu'ux u yaantal je'el máax iknale pero leti'e u tuukul beyo le káan weenek le u yuumil le najo ku yokoltik jump'éel paal wáaj jump'éel jun nojoch máak wáaj bix beyo pero yaan u tsa'aysik k'oja'anil ti ku k'áatik u páajtalil ti jump'éel máak tu yotoche"""

fonemas = tokenizer.tokenize(fragmento)
print("palabras:", len(fragmento.split()))
print("fonemas :", len([f for f in fonemas.split() if f != "SIL"]))
print()
print(fonemas[:400], "...")

palabras: 128
fonemas : 487

X T A K U M B I L SIL X U N AA_H N SIL L E SIL TS I K B A L O SIL U SIL K_G AA_L B A_G E SIL X T A_RG K U M B I L SIL X U N AA_H N SIL X T A_RG K U M B I L SIL X U N AA_H N E SIL U SIL K_G AA_H T SIL U SIL Y A_RG L E SIL T U_RG X SIL T A_RG K A_RG N SIL J U M P_G EE_H L SIL K O_RG L E L SIL J AA_L J SIL T A_RG K U M B I L SIL X U N AA_H N O SIL L E SIL X U N AA_H N O SIL J U M P_G EE_H L SIL K O_R ...
